# Scikit-learn - Pipelines

Este notebook cubre el uso de **pipelines** en scikit-learn: encadenar preprocesamiento y modelo en un solo estimador, evitar data leakage, usar `ColumnTransformer` para columnas numéricas y categóricas, y combinar pipelines con búsqueda de hiperparámetros (`GridSearchCV`).

## ¿Por qué usar Pipelines?

- **Un solo objeto**: `fit` y `predict` aplican todos los pasos en orden.
- **Evitar data leakage**: el escalado/imputación se ajusta solo con datos de entrenamiento.
- **Reproducibilidad**: mismo flujo para entrenamiento y producción.
- **Búsqueda de hiperparámetros**: ajustar parámetros de transformadores y del modelo a la vez.

## Pipeline básico

`Pipeline` recibe una lista de tuplas `(nombre, estimador)`. Cada paso (excepto el último) debe tener `fit_transform`; el último puede ser un predictor (`fit`, `predict`).

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
import numpy as np

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline: imputar → escalar → clasificar
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=200))
])

pipe.fit(X_train, y_train)
score = pipe.score(X_test, y_test)
print(f"Precisión: {score:.4f}")

# predict aplica imputer → scaler → clf.predict automáticamente
y_pred = pipe.predict(X_test)

Precisión: 1.0000


## Acceder a pasos del pipeline

Por nombre o por índice. Útil para inspeccionar o cambiar parámetros.

In [2]:
# Por nombre
pipe['scaler']
pipe.named_steps['clf'].coef_.shape

# Por índice
pipe[0]   # SimpleImputer
pipe[-1]  # LogisticRegression

# Listar pasos
pipe.steps

[('imputer', SimpleImputer()),
 ('scaler', StandardScaler()),
 ('clf', LogisticRegression(max_iter=200, random_state=42))]

## ColumnTransformer: distintas transformaciones por tipo de columna

Cuando tienes columnas **numéricas** y **categóricas**, aplicas un preprocesamiento distinto a cada grupo. `ColumnTransformer` aplica cada transformación a las columnas indicadas y concatena el resultado.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd

# DataFrame de ejemplo: numéricas + categóricas
df = pd.DataFrame({
    'edad': [25, 30, np.nan, 35],
    'ingresos': [30000, 45000, 50000, 40000],
    'ciudad': ['A', 'B', 'A', 'B'],
    'genero': ['M', 'F', 'M', 'F']
})
X = df

num_cols = ['edad', 'ingresos']
cat_cols = ['ciudad', 'genero']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

X_trans = preprocessor.fit_transform(X)
print("Shape tras ColumnTransformer:", X_trans.shape)
print("Columnas (num escaladas + cat one-hot):", preprocessor.get_feature_names_out())

Shape tras ColumnTransformer: (4, 4)
Columnas (num escaladas + cat one-hot): ['num__edad' 'num__ingresos' 'cat__ciudad_B' 'cat__genero_M']


## Pipeline completo: ColumnTransformer + modelo

Se suele poner el `ColumnTransformer` como primer paso del `Pipeline` y el modelo como último.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier

# make_pipeline no exige nombres; se generan automáticamente (lowercase del tipo)
pipe_full = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=50, random_state=42))
])

# Alternativa con make_pipeline (nombres automáticos)
pipe_full_alt = make_pipeline(preprocessor, RandomForestClassifier(n_estimators=50, random_state=42))

pipe_full.fit(X, y=np.array([0, 1, 0, 1]))  # etiquetas de ejemplo
pipe_full.predict(X)

array([0, 1, 0, 1])

## GridSearchCV con pipeline

Los hiperparámetros de los pasos se referencian con `nombre_paso__parametro`. Así puedes tunear preprocesamiento y modelo a la vez.

In [5]:
from sklearn.model_selection import GridSearchCV

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=500))
])

param_grid = {
    'scaler__with_mean': [True, False],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__solver': ['lbfgs', 'saga']
}

search = GridSearchCV(pipe, param_grid, cv=3, scoring='accuracy')
search.fit(X_train, y_train)

print("Mejores parámetros:", search.best_params_)
print(f"Mejor CV score: {search.best_score_:.4f}")
print(f"Score en test: {search.score(X_test, y_test):.4f}")

# El mejor estimador ya es el pipeline ajustado con los mejores parámetros
best_pipe = search.best_estimator_

/home/sergio/Documentos/Curso IA y Big Data/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sergio/Documentos/Curso IA y Big Data/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sergio/Documentos/Curso IA y Big Data/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sergio/Documentos/Curso IA y Big Data/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sergio/Documentos/Curso IA y Big Data/cheat-sheets-ia/.venv/lib/python3.12

Mejores parámetros: {'clf__C': 1.0, 'clf__solver': 'lbfgs', 'scaler__with_mean': True}
Mejor CV score: 0.9583
Score en test: 1.0000


/home/sergio/Documentos/Curso IA y Big Data/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Resumen

| Concepto | Uso |
|----------|-----|
| `Pipeline([('nombre', est), ...])` | Encadenar pasos; último paso puede ser predictor. |
| `make_pipeline(est1, est2, ...)` | Igual que Pipeline pero con nombres automáticos. |
| `pipe.fit(X, y)` / `pipe.predict(X)` | Entrena o predice aplicando todos los pasos. |
| `pipe['nombre']` o `pipe[n]` | Acceder a un paso por nombre o índice. |
| `ColumnTransformer([('nombre', trans, cols), ...])` | Distintas transformaciones por columnas. |
| `param_grid = {'paso__param': [valores]}` | En GridSearchCV, parámetros de cada paso del pipeline. |